## Clustering: Análisis

- Se comparan las técnicas, se interpretan las tipologías y los outliers

- Se responde la PQ1 (concentración y cautividad)

### Constantes

- `RANGO_HHI`: umbrales de HHI para llamar "cautiva" a una entidad

- `RANGO_CONSENSO`: cuántos métodos deben marcar a un actor para considerarlo outlier

- `P_LEJANO`: percentil de distancia al centroide desde el que KMeans marca un actor como outlier

In [1]:
RANGO_HHI = [0.25, 0.40, 0.50]
RANGO_CONSENSO = [1, 2, 3]
P_LEJANO = 0.95

### Utilidades

In [2]:
NOTEBOOK = "e_analisis"
%run ../funciones.ipynb
%run funciones.ipynb

In [3]:
spark = crear_spark(NOTEBOOK)
verificar_cluster(spark)

Master        : spark://spark-master:7077
Executors     : 2
Cores totales : 2
Memoria driver: 2.0 GB
UI del job    : http://localhost:4040


In [4]:
config = leer_config()
tabla = lambda nb, t: pd.read_parquet(proyecto / "02_Clustering" / "temp" / nb / "tablas" / t)

res = {}
for s, c in config.items():
    d, X = cargar_local(s, c["n_comp"])
    d = (d.merge(pd.read_parquet(datos_dir / f"densidad_{s}"), on=c["key"])
          .merge(pd.read_parquet(datos_dir / f"jerarquico_{s}")[[c["key"], "jerarquico"]], on=c["key"], how="left"))
    res[s] = (d, X)
    print(f"{s}: {len(d):,} actores | KMeans k = {c['k']}")

# los proveedores no traen dirección en el OCDS: su región es la de sus clientes, la de mayor monto
hechos = pd.read_parquet(final_dir / "hechos", columns=["entidad_key", "proveedor_key", "region", "monto_asignado"])
reg = (hechos.groupby(["proveedor_key", "region"]).monto_asignado.sum().reset_index()
             .sort_values("monto_asignado", ascending=False).drop_duplicates("proveedor_key")[["proveedor_key", "region"]])
res["proveedor"] = (res["proveedor"][0].merge(reg, on="proveedor_key", how="left"), res["proveedor"][1])

entidad: 2,696 actores | KMeans k = 4
proveedor: 13,317 actores | KMeans k = 4


---
### **1. Técnicas**

In [5]:
def fila(s, metodo, lab, X, donde, params):
    lab = np.asarray(lab)
    return {"sujeto": s, "metodo": metodo, "parametros": params, "corre_en": donde, "n": int((lab > -2).sum()),
            "n_clusters": len(set(lab) - {-1}), "pct_ruido": round(100 * (lab == -1).mean(), 1), "silueta": round(silueta(X, lab), 3)}

tecnicas = []
for s, (d, X) in res.items():
    c = config[s]; j = d.jerarquico.notna().to_numpy()
    tecnicas += [fila(s, "KMeans++", d.kmeans, X, "Spark", f"k={c['k']}"),
                 fila(s, "DBSCAN", d.dbscan, X, "local", f"eps={c['dbscan']['eps']:.2f}, min_samples={c['dbscan']['min_samples']}"),
                 fila(s, "HDBSCAN", d.hdbscan, X, "local", f"min_cluster_size={c['hdbscan']['min_cluster_size']}, min_samples={c['hdbscan']['min_samples']}"),
                 fila(s, "Jerárquico", d.jerarquico[j].astype(int), X[j], "local (muestra)", f"{c['jerarquico']['enlace']}, k={c['jerarquico']['k']}")]
tecnicas = pd.DataFrame(tecnicas)
tecnicas

,sujeto,metodo,parametros,corre_en,n,n_clusters,pct_ruido,silueta
0,entidad,KMeans++,k=4,Spark,2696,4,0.0,0.203
1,entidad,DBSCAN,"eps=1.74, min_samples=5",local,2696,2,2.6,0.438
2,entidad,HDBSCAN,"min_cluster_size=10, min_samples=1",local,2696,2,18.8,0.141
3,entidad,Jerárquico,"ward, k=4",local (muestra),2696,4,0.0,0.122
4,proveedor,KMeans++,k=4,Spark,13317,4,0.0,0.337
5,proveedor,DBSCAN,"eps=1.18, min_samples=5",local,13317,6,2.6,0.291
6,proveedor,HDBSCAN,"min_cluster_size=100, min_samples=5",local,13317,6,27.0,0.211
7,proveedor,Jerárquico,"ward, k=4",local (muestra),5000,4,0.0,0.346


In [6]:
metodos = ["kmeans", "dbscan", "hdbscan", "jerarquico"]
ari_metodos = pd.concat({s: pd.DataFrame([[round(ari(d.loc[d.jerarquico.notna(), a].astype(int), d.loc[d.jerarquico.notna(), b].astype(int)), 2)
                                            for b in metodos] for a in metodos], index=metodos, columns=metodos)
                         for s, (d, X) in res.items()})
ari_metodos

kmeans  dbscan  hdbscan  jerarquico
entidad   kmeans        1.00    0.00     0.01        0.36
          dbscan        0.00    1.00     0.00        0.00
          hdbscan       0.01    0.00     1.00        0.01
          jerarquico    0.36    0.00     0.01        1.00
proveedor kmeans        1.00    0.01     0.46        0.54
          dbscan        0.01    1.00     0.00        0.02
          hdbscan       0.46    0.00     1.00        0.65
          jerarquico    0.54    0.02     0.65        1.00

- **KMeans++ (Spark)** da 4 tipologías interpretables en ambos sujetos, sin ruido

- **DBSCAN** deja casi todo en 1 grupo grande (2 grupos en entidades, 6 en proveedores) con 2.6% de ruido: 

- Su silueta alta (0.44) refleja un grupo enorme, no tipologías

- **HDBSCAN** marca 19-27% de ruido: la densidad es pareja y no hay islas separadas

- **Ward** (muestra de 5,000 proveedores) reproduce los perfiles de KMeans: ARI 0.54 en proveedores y 0.36 en entidades

---
### **2. Tipologías - KMeans**

In [7]:
perfiles = {}
for s, (d, X) in res.items():
    monto = "monto_adjudicado_total" if s == "entidad" else "monto_total"
    g = d.groupby("kmeans")
    p = pd.DataFrame({"n": g.size(), "pct_actores": (100 * g.size() / len(d)).round(1),
                      "pct_monto": (100 * g[monto].sum() / d[monto].sum()).round(1),
                      "region_principal": g.region.agg(lambda r: r.value_counts().index[0] if r.notna().any() else None),
                      "pct_lima": g.region.apply(lambda r: round(100 * (r == "LIMA").mean(), 1))})
    if s == "proveedor":
        p = p.join(pd.crosstab(d.kmeans, d.tipo, normalize="index").mul(100).round(1).add_prefix("pct_"))
    perfiles[s] = p.join(perfil_z(d, config[s]["feats"], "kmeans")[["rasgos"]])
display(perfiles["entidad"]); display(perfiles["proveedor"])

,n,pct_actores,pct_monto,region_principal,pct_lima,rasgos
kmeans,,,,,,
0,909,33.7,2.4,PUNO,6.7,"log_monto_total ↓0.8σ, log_monto_mediano ↓0.7σ, log_n_procesos ↓0.7σ"
1,234,8.7,13.8,LIMA,44.0,"pct_postor_unico ↑2.4σ, pct_directa ↑2.2σ, log_postores_promedio ↓0.8σ"
2,680,25.2,6.6,ANCASH,5.7,"pct_obras ↑1.3σ, log_monto_mediano ↑0.8σ, pct_bienes ↓0.7σ"
3,873,32.4,77.2,LIMA,24.3,"log_n_procesos ↑1.0σ, log_monto_total ↑0.8σ, share_top_proveedores ↓0.7σ"


,n,pct_actores,pct_monto,region_principal,pct_lima,pct_CONS,pct_EXT,pct_PN,pct_RUC,rasgos
kmeans,,,,,,,,,,
0,2394,18.0,50.4,LIMA,50.5,0.0,0.0,8.9,91.1,"log_n_entidades ↑1.6σ, meses_activos ↑1.5σ, log_n_regiones ↑1.4σ"
1,8252,62.0,14.5,LIMA,29.4,0.0,0.4,21.8,77.8,"pct_postor_unico ↓0.4σ, meses_activos ↓0.3σ, log_n_entidades ↓0.3σ"
2,1650,12.4,2.7,LIMA,79.2,0.0,10.5,52.5,36.9,"pct_postor_unico ↑2.2σ, tasa_exito ↑1.8σ, pct_directa ↑1.0σ"
3,1021,7.7,32.4,LIMA,40.6,1.3,0.5,3.3,94.9,"pct_obras ↑2.7σ, log_monto_mediano ↑2.0σ, log_monto_total ↑1.3σ"


#### a. Entidades

- **grandes diversificadas** (32% de entidades, **77% del monto**): muchas compras y muchos proveedores

- **compradoras pequeñas** (34%, 2% del monto): pocas compras y de bajo monto, la mayoría fuera de Lima

- **compradoras de obras** (25%)

- **compra sin competencia** (9%, con 44% en Lima): contratación directa y postor único muy altos

#### b. Proveedores

- **grandes multicliente** (18%, **50% del monto**): venden a muchas entidades y regiones

- **típicos de pocos clientes** (62%)

- **ganadores sin competencia** (12%, con 79% en Lima): 53% personas naturales y 11% extranjeros

- **contratistas de obras** (8%, 32% del monto)

In [8]:
NOMBRES = {"entidad": {0: "compradoras pequeñas", 1: "compra sin competencia", 2: "compradoras de obras",
                       3: "grandes diversificadas"},
           "proveedor": {0: "grandes multicliente", 1: "típicos de pocos clientes", 2: "ganadores sin competencia",
                         3: "contratistas de obras"}}
for s in perfiles:
    perfiles[s].insert(0, "tipologia", perfiles[s].index.map(NOMBRES[s]))

---
### **3. Concentración y cautividad**

#### a. Umbral de HHI

In [9]:
ent = res["entidad"][0]
cautivas = pd.DataFrame([{"umbral_hhi": t, "entidades": int((ent.hhi_proveedores >= t).sum()),
                          "pct_entidades": round(100 * (ent.hhi_proveedores >= t).mean(), 1),
                          "postor_unico_med": round(ent.loc[ent.hhi_proveedores >= t, "pct_postor_unico"].median(), 3),
                          "directa_med": round(ent.loc[ent.hhi_proveedores >= t, "pct_directa"].median(), 3),
                          "procesos_med": ent.loc[ent.hhi_proveedores >= t, "n_procesos"].median()} for t in RANGO_HHI])
cautivas

,umbral_hhi,entidades,pct_entidades,postor_unico_med,directa_med,procesos_med
0,0.25,807,29.9,0.0,0.0,12.0
1,0.40,332,12.3,0.0,0.0,11.0
2,0.50,191,7.1,0.0,0.0,10.0


- HHI ≥ 0.25: 807 entidades (29.9%); ≥ 0.40: 12.3%; ≥ 0.50: 7.1%

- En los tres umbrales la mediana de postor único y de contratación directa es 0

- La cautividad se explica por depender del monto de unos pocos proveedores

- Se fija `UMBRAL_HHI = 0.25`: es el umbral usual de alta concentración en competencia

In [10]:
ent["cautiva"] = (ent.hhi_proveedores >= UMBRAL_HHI).astype(int)
por_tipo = (ent.groupby("kmeans").agg(entidades=("cautiva", "size"), cautivas=("cautiva", "sum"))
               .assign(pct_cautivas=lambda t: (100 * t.cautivas / t.entidades).round(1)))
por_tipo.insert(0, "tipologia", por_tipo.index.map(NOMBRES["entidad"]))
por_tipo

,tipologia,entidades,cautivas,pct_cautivas
kmeans,,,,
0,compradoras pequeñas,909,430,47.3
1,compra sin competencia,234,58,24.8
2,compradoras de obras,680,292,42.9
3,grandes diversificadas,873,27,3.1


#### b. Quién atiende a las entidades cautivas

Para cada entidad cautiva, su proveedor principal y la tipología de ese proveedor

In [11]:
top = (hechos.groupby(["entidad_key", "proveedor_key"]).monto_asignado.sum().reset_index()
             .sort_values("monto_asignado", ascending=False).drop_duplicates("entidad_key"))
prov = res["proveedor"][0][["proveedor_key", "kmeans"]].rename(columns={"kmeans": "tipo_proveedor"})
atienden = (ent.loc[ent.cautiva == 1, ["entidad_key"]].merge(top, on="entidad_key").merge(prov, on="proveedor_key", how="left"))
atienden["tipologia_proveedor"] = atienden.tipo_proveedor.map(NOMBRES["proveedor"]).fillna("esporádico (fuera del clustering)")
atienden_res = atienden.tipologia_proveedor.value_counts().rename("entidades").to_frame()
atienden_res["pct"] = (100 * atienden_res.entidades / atienden_res.entidades.sum()).round(1)
atienden_res

,entidades,pct
tipologia_proveedor,,
esporádico (fuera del clustering),612,75.8
contratistas de obras,71,8.8
grandes multicliente,64,7.9
típicos de pocos clientes,56,6.9
ganadores sin competencia,4,0.5


- **Dónde están las cautivas:** 47% de las compradoras pequeñas y 43% de las de obras son cautivas

- Contra solo 3% de las grandes diversificadas. La cautividad es sobre todo de **entidades chicas y de obras**

- **Quién las atiende:** el **76% de las entidades cautivas depende de un proveedor "esporádico"** 

- Menos de 3 procesos en 3 años, y el **89% de esos son consorcios** de un solo proceso

- La dependencia típica no es de un gran proveedor recurrente, sino de **un consorcio armado para una obra o contrato grande**

---
### **4. Outliers**

Un actor se marca por KMeans (distancia al centroide sobre el percentil `P_LEJANO`), DBSCAN (ruido) o HDBSCAN (ruido)

In [12]:
marcas = {}
for s, (d, X) in res.items():
    lejano = d.groupby("kmeans").dist_centro.transform(lambda x: x >= x.quantile(P_LEJANO))
    marcas[s] = pd.DataFrame({"kmeans": lejano.astype(int), "dbscan": (d.dbscan == -1).astype(int),
                              "hdbscan": (d.hdbscan == -1).astype(int)}).assign(votos=lambda m: m.sum(axis=1))
consenso = pd.DataFrame([{"sujeto": s, "min_metodos": c, "outliers": int((m.votos >= c).sum()), "pct": round(100 * (m.votos >= c).mean(), 1)}
                         for s, m in marcas.items() for c in RANGO_CONSENSO])
consenso.pivot(index="min_metodos", columns="sujeto", values=["outliers", "pct"])

outliers               pct          
sujeto       entidad proveedor entidad proveedor
min_metodos                                     
1              525.0    3728.0    19.5      28.0
2              149.0     726.0     5.5       5.5
3               37.0     149.0     1.4       1.1

- **≥ 1 método:** 19.5% de entidades y 28.0% de proveedores; casi todo es ruido de HDBSCAN

- **≥ 2 métodos:** 5.5% en ambos

- **≥ 3 métodos:** 1.1-1.4%

- Se fija `CONSENSO = 2`: con 1, el ruido de HDBSCAN infla la lista; con 3, quedan solo los casos extremos

In [13]:
outliers = []
for s, (d, X) in res.items():
    monto = "monto_adjudicado_total" if s == "entidad" else "monto_total"
    o = d.loc[marcas[s].votos >= CONSENSO].assign(votos=marcas[s].votos)
    o["tipologia"] = o.kmeans.map(NOMBRES[s])
    o["rasgos"] = [rasgos(z, config[s]["feats"]) for z in o.z]
    outliers.append(o.rename(columns={config[s]["key"]: "clave", monto: "monto"})
                     .assign(sujeto=s)[["sujeto", "clave", "nombre", "tipologia", "votos", "n_procesos", "monto", "dist_centro", "rasgos"]])
outliers = pd.concat(outliers).sort_values(["sujeto", "votos", "dist_centro"], ascending=[True, False, False])
outliers.groupby("sujeto").head(8)

,sujeto,clave,nombre,tipologia,votos,n_procesos,monto,dist_centro,rasgos
2194,entidad,ENT:205250,AUTORIDAD NACIONAL DE CONTROL DEL MINISTERIO PÚBLICO,compra sin competencia,3,5,8.465000e+05,9.702132,"pct_directa ↑9.6σ, pct_postor_unico ↑7.3σ, log_postores_promedio ↓4.1σ"
1364,entidad,ENT:205170,FORTALECIMIENTO DEL SISTEMA NACIONAL DE VIGILANCIA EN SALUD PÚBLICA,compra sin competencia,3,8,1.410137e+05,8.096230,"pct_postor_unico ↑7.3σ, log_monto_mediano ↓5.8σ, log_postores_promedio ↓4.1σ"
2657,entidad,ENT:202125,UNIDAD EJECUTORA 411 SALUD JULCAN,compra sin competencia,3,6,1.347605e+06,8.054921,"pct_directa ↑9.6σ, pct_postor_unico ↑4.7σ, log_postores_promedio ↓3.7σ"
2094,entidad,ENT:86,MUNICIPALIDAD DISTRITAL DE SOLOCO,compra sin competencia,3,5,3.371322e+06,7.590630,"pct_directa ↑9.6σ, pct_obras ↑3.6σ, log_postores_promedio ↓3.5σ"
1515,entidad,ENT:204950,PROGRAMA NACIONAL DE INVESTIGACIÓN CIENTÍFICA Y ESTUDIOS AVANZADOS,compradoras pequeñas,3,62,4.207181e+06,6.968740,"log_monto_mediano ↓8.8σ, pct_postor_unico ↑3.2σ, pct_bienes ↓1.6σ"
2640,entidad,ENT:200366,PROYECTO DE MODERNIZACION DE LA GESTION DE LOS RECURSOS HIDRICOS,compra sin competencia,3,12,2.867238e+07,6.890773,"pct_directa ↑7.9σ, pct_postor_unico ↑4.7σ, log_monto_mediano ↑3.7σ"
2271,entidad,ENT:74,MUNICIPALIDAD DISTRITAL DE HUANCAS - AMAZONAS,compra sin competencia,3,5,1.103210e+07,6.794252,"pct_directa ↑5.6σ, pct_postor_unico ↑4.1σ, share_top_proveedores ↑2.9σ"
2171,entidad,ENT:203748,UNIDAD EJECUTORA 005: VENTANILLA UNICA DE COMERCIO EXTERIOR - SEGUNDA ETAPA,compra sin competencia,3,5,1.491763e+05,6.709942,"pct_postor_unico ↑7.3σ, log_postores_promedio ↓4.1σ, log_monto_total ↓2.7σ"
9489,proveedor,RUC:20501887286,DIAGNOSTICA PERUANA S.A.C.,grandes multicliente,3,572,4.083834e+08,7.981661,"meses_activos ↑6.0σ, log_n_regiones ↑5.1σ, log_n_entidades ↑5.0σ"
1581,proveedor,RUC:20100028698,FERREYROS SOCIEDAD ANÓNIMA,grandes multicliente,3,235,5.844381e+08,7.975726,"meses_activos ↑5.8σ, log_n_entidades ↑5.7σ, log_n_regiones ↑4.6σ"


- **Entidades** (149): sobre todo del tipo "compra sin competencia", con 100% de contratación directa en pocos procesos

- **Proveedores** (726): los **grandes nacionales**

- Los outliers son de dos tipos: **compras sin competencia** (entidades) y **proveedores de escala nacional**

---
### **5. Variación de los resultados**

Se varían parámetros y se evaluan resultados

In [14]:
curvas = tabla("b_kmeans", "curvas_k")
variacion = pd.concat([
    curvas.assign(parametro="k (KMeans)", valor=curvas.k.astype(str), resultado="silueta " + curvas.silueta.round(3).astype(str))
          [lambda t: t.k.isin([config[s]["k"] + dk for s in config for dk in (-1, 0, 1)])],
    tabla("c_densidad", "topes_ruido").assign(parametro="ruido máximo (DBSCAN)", valor=lambda t: t.ruido_max.astype(str),
                                             resultado=lambda t: t.n_clusters.astype(int).astype(str) + " clústeres, " + t.pct_ruido.astype(str) + "% ruido"),
    cautivas.assign(sujeto="entidad", parametro="umbral HHI (cautiva)", valor=cautivas.umbral_hhi.astype(str),
                    resultado=cautivas.pct_entidades.astype(str) + "% de entidades")])[["sujeto", "parametro", "valor", "resultado"]]
variacion

,sujeto,parametro,valor,resultado
1,entidad,k (KMeans),3,silueta 0.306
2,entidad,k (KMeans),4,silueta 0.334
3,entidad,k (KMeans),5,silueta 0.305
10,proveedor,k (KMeans),3,silueta 0.435
11,proveedor,k (KMeans),4,silueta 0.516
12,proveedor,k (KMeans),5,silueta 0.346
0,entidad,ruido máximo (DBSCAN),0.05,"2 clústeres, 2.6% ruido"
1,entidad,ruido máximo (DBSCAN),0.1,"2 clústeres, 2.6% ruido"
2,entidad,ruido máximo (DBSCAN),0.2,"2 clústeres, 2.6% ruido"
3,proveedor,ruido máximo (DBSCAN),0.05,"6 clústeres, 2.6% ruido"


- **k:** con k = 3 o 5 la silueta baja (0.31 y 0.31 en entidades; 0.44 y 0.35 en proveedores)

- k = 4 es el único máximo común

- **Ruido de DBSCAN:** la elección no cambia entre 5% y 10%; con 20% en proveedores pasaría a 2 grupos

- **HHI:** el porcentaje de entidades cautivas va de 30% a 7% según el umbral

---
### **6. Casos concretos**

In [15]:
casos = pd.concat([
    ent.loc[ent.cautiva == 1].nlargest(1, "monto_adjudicado_total").assign(caso="entidad cautiva de mayor monto", sujeto="entidad"),
    ent.nlargest(1, "dist_centro").assign(caso="entidad más atípica", sujeto="entidad"),
    res["proveedor"][0].nlargest(1, "n_entidades").assign(caso="proveedor con más entidades", sujeto="proveedor")])
casos["tipologia"] = [NOMBRES[s][k] for s, k in zip(casos.sujeto, casos.kmeans)]
casos[["caso", "nombre", "tipologia", "n_procesos", "hhi_proveedores", "hhi_entidades", "dist_centro", "dbscan", "hdbscan"]]

,caso,nombre,tipologia,n_procesos,hhi_proveedores,hhi_entidades,dist_centro,dbscan,hdbscan
301,entidad cautiva de mayor monto,GOBIERNO REGIONAL DE LORETO - ORGANISMO PUBLICO INFRAESTRUCTURA PARA LA PROD...,compra sin competencia,93,0.422794,NaN,3.747299,-1,-1
2194,entidad más atípica,AUTORIDAD NACIONAL DE CONTROL DEL MINISTERIO PÚBLICO,compra sin competencia,5,0.253595,NaN,9.702132,-1,-1
2990,proveedor con más entidades,GRUPO SANTA FE SOCIEDAD ANONIMA CERRADA - GRUPO SANTA FE S.A.C.,grandes multicliente,1704,NaN,0.041227,7.149419,-1,-1


- **GR Loreto, Organismo de Infraestructura** (entidad cautiva de mayor monto): S/1,548M en 93 procesos, 

- **64% del monto a un solo consorcio** ("Consorcio San Juan")

- **Autoridad Nacional de Control del MP** (entidad atipica): 5 procesos, todos por contratación directa y con postor único

- **Grupo Santa Fe** (proveedor con más entidades): 1,704 procesos con 275 entidades

---
### **7. Limitaciones y ética**

- **Señal, no acusación:** una entidad cautiva o un outlier se aparta del patrón; puede tener un mandato singular

- **Consorcios sin miembros:** el 89% de la dependencia de las cautivas recae en consorcios de un proceso

- **Cola larga:** el 84.5% de los proveedores (los esporádicos) queda fuera del clustering

- **Continuo, no islas:** los métodos de densidad muestran que las tipologías son cortes de una masa continua

---
### **8. Síntesis**

- **4 tipologías** en entidades y en proveedores, estables entre métodos y entre semillas

- El gasto se concentra en pocas tipologías: las **grandes diversificadas** hacen el 77% del gasto de entidades

- Los **grandes multicliente** reciben el 50% del monto de proveedores

- **PQ1:** el 30% de las entidades es cautiva, sobre todo **pequeñas y de obras**,

- Su dependencia recae en **consorcios de un solo proceso**, no en grandes proveedores recurrentes

- **Outliers** (consenso ≥ 2): compras 100% directas en entidades y proveedores de escala nacional (a muchas entidades)


---
### **9. Escritura de resultados**

In [16]:
escritos = guardar_tablas({
    "tecnicas": tecnicas, "ari_metodos": ari_metodos.reset_index().rename(columns={"level_0": "sujeto", "level_1": "metodo"}),
    "tipologias": pd.concat({s: p.reset_index() for s, p in perfiles.items()}).reset_index(level=0).rename(columns={"level_0": "sujeto"}),
    "cautivas_umbral": cautivas, "cautivas_por_tipologia": por_tipo.reset_index(), "atienden_cautivas": atienden_res.reset_index(),
    "consenso_outliers": consenso, "outliers": outliers, "variacion": variacion})

Escrito en 02_Clustering/temp/e_analisis/: tecnicas, ari_metodos, tipologias, cautivas_umbral, cautivas_por_tipologia, atienden_cautivas, consenso_outliers, outliers, variacion


#### a. Verificación

In [17]:
verificar_salidas(escritos)

OK: 9 tablas (parquet + json)


---
### **10. Cerrar sesión**

In [18]:
spark.stop()